## Set root

In [46]:
import pyrootutils
from pathlib import Path

root = pyrootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
root_iris = root / "iris"

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load data module
The saved data module contains the same train/validation/test splits used when the RandomForest model was trained.

In [47]:
import numpy as np
from iris.modules.sklearn_iris_datamodule import IrisDataModule

data_module = IrisDataModule(
    data_dir=root_iris / "metadata/data",
    use_onehot_encoder=True,
    standardize=True,
).from_joblib()

print("Train shapes:", data_module.data_train[0].shape, data_module.data_train[1].shape)
print("Validation shapes:", data_module.data_val[0].shape, data_module.data_val[1].shape)
print("Test shapes:", data_module.data_test[0].shape, data_module.data_test[1].shape)
print("Classes:", data_module.classes)

Data loaded from C:\Julia_Projects\safetycage-tutorials\iris\metadata\data\iris.npz.
Train shapes: (96, 4) (96, 3)
Validation shapes: (24, 4) (24, 3)
Test shapes: (30, 4) (30, 3)
Classes: {0: 'setosa', 1: 'versicolor', 2: 'virginica'}


# Load prediction model

In [48]:
import joblib

model = joblib.load(root_iris / "metadata/model/random_forest.joblib")
model

RandomForestClassifier(n_estimators=300, random_state=42)

## Instantiate the model module
The model module translates sklearn model outputs into the standardized interface expected by SafetyCage.

In [49]:
from iris.modules.sklearn_iris_modelmodule import SklearnIrisModelModule

model_module = SklearnIrisModelModule(
    model=model,
    selected_layers=["input"],
    last_layer="input",
    use_onehot_encoder=True,
)

print(model_module.model_shape)

{'input': 4, 'probabilities': 3, 'log_probabilities': 3}


In [50]:
inputs_train, labels_train = data_module.data_train

predictions_train = model_module._get_predictions(inputs_train)
predictions_train[:5]

array([[1., 0., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])

In [51]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

accuracy = accuracy_score(np.argmax(labels_train, axis=1), np.argmax(predictions_train, axis=1))
precision = precision_score(np.argmax(labels_train, axis=1), np.argmax(predictions_train, axis=1), average="weighted", zero_division=0)
recall = recall_score(np.argmax(labels_train, axis=1), np.argmax(predictions_train, axis=1), average="weighted", zero_division=0)

print("Performance on training data:")
print(f"Accuracy: {accuracy * 100:.4f} %")
print(f"Precision: {precision * 100:.4f} %")
print(f"Recall: {recall * 100:.4f} %")

Performance on training data:
Accuracy: 100.0000 %
Precision: 100.0000 %
Recall: 100.0000 %


## Instantiate a safetycage

In [52]:
from safetycage_testing.methods.msp import MSP

safetycage = MSP(model_module, data_module)

## Train the safetycage

In [53]:
# Train the safety cage on training data
# safetycage.train_cage(x=inputs_train, y=labels_train, y_pred=predictions_train)
safetycage.train_cage()

## Safetycage predicts on validation data
Safetycage predicts whether samples have been misclassified by the classifier.

In [54]:
inputs_val, labels_val = data_module.data_val

predictions_val = model_module._get_predictions(inputs_val)
incorrect_predictions_val = ~(
    np.argmax(predictions_val, axis=1) == np.argmax(labels_val, axis=1)
)

max_probs_val = safetycage.predict(
    x=inputs_val,
    y=predictions_val,
)

max_probs_val[:5]

array([1.        , 0.97333333, 0.97666667, 0.67      , 1.        ])

## Find threshold that maximizes a performance metric

In [55]:
from safetycage_testing.utils.evaluate import find_best_threshold, MCC, accuracy

metric = MCC

optimisation_result = find_best_threshold(
    y_probs=max_probs_val,
    y_true=incorrect_predictions_val,
    metric_fn=metric,
    greater_is_better=True,
)

threshold_val = optimisation_result["alpha_opt"]
metric_val = optimisation_result["metric_max"]

print("\nOptimal threshold and performance metric based on validation data:")
print(f"Threshold = {threshold_val:.4f}")
print(f"{metric.__name__} = {metric_val:.4f}")

safetycage.alpha = threshold_val
safetycage.threshold_metric_val = metric_val

threshold_flag_result = safetycage.find_best_threshold_flag(
    y_probs=max_probs_val,
    y_true=incorrect_predictions_val,
    metric_fn=metric
)

print("\nOptimal threshold and performance metric using safetycage.flag() based on validation data:")
print(f"Flag Threshold = {threshold_flag_result['alpha_opt']:.4f}")
print(f"{metric.__name__} = {threshold_flag_result['metric_max']:.4f}")


Optimal threshold and performance metric based on validation data:
Threshold = 0.8767
MCC = 0.4663

Optimal threshold and performance metric using safetycage.flag() based on validation data:
Flag Threshold = 0.8768
MCC = 0.4663


## Test with optimal threshold on the test partition

In [56]:
inputs_test, labels_test = data_module.data_test

prediction_test = model_module._get_predictions(inputs_test)
incorrect_prediction_test = ~(
    np.argmax(prediction_test, axis=1) == np.argmax(labels_test, axis=1)
)

max_probs_test = safetycage.predict(
    x=inputs_test,
    y=prediction_test,
)

flags_test = safetycage.flag(max_probs_test)
flags_test[:5]

array([False, False, False, False, False])

# Calculate metrics on the test partition

In [57]:
import json
from pathlib import Path

from safetycage_testing.utils.evaluate import (
    calculate_confusion_rates,
    calculate_metrics,
    calculate_auroc,
)

results_dir = Path("./results")
results_dir.mkdir(parents=True, exist_ok=True)

confusion_rates_test = calculate_confusion_rates(
    y=incorrect_prediction_test,
    y_pred=flags_test,
)

auroc_test = calculate_auroc(
    safetycage=safetycage,
    y_true=incorrect_prediction_test,
    y_scores=max_probs_test,
)

metrics_test = calculate_metrics(y=incorrect_prediction_test, y_pred=flags_test)
metrics_test.update(confusion_rates_test)
metrics_test.update({"AUROC": auroc_test})

with open(results_dir / "metrics_test.json", "w") as f:
    json.dump(metrics_test, f, indent=4)

for name, value in metrics_test.items():
    print(f"{name}: {value:.4f}")

Precision: 0.2000
Recall: 0.5000
Specificity: 0.8571
NPV: 0.9600
MCC: 0.2390
Accuracy: 0.8333
F1-score: 0.2857
TP: 1.0000
TN: 24.0000
FP: 4.0000
FN: 1.0000
AUROC: 0.8393


## Plots

In [58]:
from safetycage_testing.utils.evaluate import calculate_roc_curve
from safetycage_testing.utils.plot_functions import (
    plot_alpha_metric_curve,
    plot_confusion_matrix,
    plot_roc_curve,
)

best_metric_dict = find_best_threshold(
    y_probs=max_probs_test,
    y_true=incorrect_prediction_test,
    metric_fn=metric,
    greater_is_better=True,
)

plot_alpha_metric_curve(
    **best_metric_dict,
    thresholds=optimisation_result["alphas"],
    scores=optimisation_result["metric_values"],
    output_path="./results",
    alpha_val=safetycage.alpha,
    metric_val=safetycage.threshold_metric_val,
    val_label_offset=(0.0, 0.10),
)

plot_confusion_matrix(
    y_true=incorrect_prediction_test,
    y_pred=flags_test,
    normalize=None,
    output_path="./results",
)

roc = calculate_roc_curve(
    safetycage=safetycage,
    y_true=incorrect_prediction_test,
    statistics=max_probs_test,
    num_thresholds=1e4,
)

plot_roc_curve(
    **roc,
    output_path="./results",
)

In [59]:
safetycage.save_cage("./saved_cage")

In [60]:
safetycage.alpha = None
safetycage.layer_params = None

safetycage.load_cage("./saved_cage")

print(safetycage.alpha)
print(safetycage.layer_params)

0.024053924609016425
{'input': {0: {}, 1: {}, 2: {}, 'setosa': {'mean': array([-1.02631371,  0.81596524, -1.30136011, -1.25205106]), 'variance': array([[0.11731624, 0.19928638, 0.00927874, 0.02076434],
       [0.19928638, 0.67756721, 0.02486968, 0.04606276],
       [0.00927874, 0.02486968, 0.00827993, 0.00599506],
       [0.02076434, 0.04606276, 0.00599506, 0.02546737]]), 'num_observations': 32}, 'versicolor': {'mean': array([ 0.11498628, -0.69381476,  0.27809887,  0.14656116]), 'variance': array([[0.33313974, 0.16769152, 0.10226875, 0.05954877],
       [0.16769152, 0.4604352 , 0.06149987, 0.05689733],
       [0.10226875, 0.06149987, 0.06587206, 0.03905264],
       [0.05954877, 0.05689733, 0.03905264, 0.0457399 ]]), 'num_observations': 32}, 'virginica': {'mean': array([ 0.91132743, -0.12215049,  1.02326124,  1.1054899 ]), 'variance': array([[0.6880634 , 0.42939023, 0.25168755, 0.109453  ],
       [0.42939023, 0.75918579, 0.16275887, 0.15383823],
       [0.25168755, 0.16275887, 0.113780